# Анализ Research Outputs

В данном файле проводится попытка анализа данных о результатах научной деятельности.
Нас будет интересовать сопоставление со схемой `swagger.json`, которую можно получить как ответ сервера при открытии со страницей спецификации (доступна в вики проекта).

## Импорт зависимостей

In [83]:
# imports
import os
import json
import pandas as pd

In [84]:
from notebooks import utils

## Чтение спецификации

In [85]:
SWAGGER_SPEC_PATH = 'notebooks/522-swagger-pretty.json'
api_spec = utils.load_from_json(SWAGGER_SPEC_PATH)

В хорошей API спецификации должно быть указано, в каком формате приходит тело ответа.
Посмотрим, что возвращает сервер при запросе получить набор из `research-outuput`.

In [86]:
api_spec['paths'][f'/research-outputs']['get']['responses']['200']['schema']

{'$ref': '#/definitions/WSResearchOutputListResult'}

Посмотрим, что возвращает сервер при запросе отдельного `research-output` по его id.

In [87]:
api_spec['paths'][f'/research-outputs/{{id}}']['get']['responses']['200']['schema']

{'$ref': '#/definitions/WSResearchOutput'}

`$ref` указывает на место в файле.
В файле спецификации можно увидеть набор объектов, находящихся под полем `definitions`.
Видимо, они определяет формат ответа для каждой сущности, приходящей при успешном ответе с сервера.

Посмотрим, как выглядит объект `research-output`, который возвращается при запросе `/research-outputs`:

In [88]:
api_spec['definitions']['WSResearchOutputListResult']['properties']

{'count': {'type': 'integer', 'format': 'int32'},
 'pageInformation': {'$ref': '#/definitions/WSPageInformation'},
 'navigationLinks': {'type': 'array',
  'xml': {'wrapped': True},
  'items': {'xml': {'name': 'navigationLink'},
   '$ref': '#/definitions/WSNavigationLink'}},
 'items': {'type': 'array',
  'xml': {'wrapped': True},
  'items': {'$ref': '#/definitions/WSResearchOutput'}}}

Видим, что все-таки вовзращается набор из объектов типа `WSResearchOutput`!
Тогда продолжим работать с ним.

## Парсинг атрибутов сущности

Надо бы написать парсер...
У каждого атрибута есть 'type': это может быть как примитив, так и ссылка на другую сущность.
Тут приходится придумать какие-то эвристики на основе того, что встречается в выборке.

In [89]:
research_schema_df = utils.fetch_schema(api_spec, 'WSResearchOutput')
research_schema_df.head()

,field,type
55,abstract,WSLocalizedString
29,additionalFiles,Array <WSElectronicVersionAdditionalFileAssoci...
30,additionalLinks,Array <WSLink>
34,articleProcessingChargeAmount,Primitive <double>
32,articleProcessingChargeAmountInArticleProcessi...,Primitive <double>


## Подсчет частот

Поскольку нет информации о том, какие поля обязательно встретятся в записи о научной деятельности, посмотрим, как часто встречается каждый из найденных в спецификации атрибутов в тестовой выгрузке.

In [90]:
researches = utils.load_from_json('../data/research-outputs.json')['items']

In [91]:
known_frequences, unknown_frequences = utils.count_field_frequences(
    items = researches, 
    keys = list(research_schema_df['field'])
)

Выведем частоту появления атрибутов из схемы в записях:

In [92]:
known_frequences_df = pd.DataFrame(known_frequences.items(), columns=['field', 'count'])
known_frequences_df.sort_values(by='count', ascending=False)

,field,count
9,category,1000
16,externalIdSource,1000
15,externalId,1000
10,confidential,1000
22,info,1000
29,openAccessPermission,1000
32,personAssociations,1000
27,managingOrganisationalUnit,1000
25,language,1000
48,title,1000


Выведем частоту появления атрибутов, которых НЕ УКАЗАНО в схеме, но они есть в выборке:

In [93]:
unknown_frequences_df = pd.DataFrame(unknown_frequences.items(), columns=['field', 'count'])
unknown_frequences_df.sort_values(by='count', ascending=False)

,field,count
1,journalAssociation,969
0,pages,873
3,volume,340
2,journalNumber,244
4,number,49
7,hostPublicationTitle,31
5,publisher,28
9,isbns,25
6,event,16
8,articleNumber,10


Соберем два датафрейма в один: соединим `research_schema_df` и `occurences_df` по колонке `field`, чтобы получить наглядное представление в формате: `field_name` - `type` - `number_of_occurences`

In [94]:
schema_fields_freqs = pd.merge(
        left=research_schema_df, 
        right=known_frequences_df, 
        on='field', 
        how='outer'
    ).sort_values(
        by='count',
        ascending=False
    )

## Результаты

Выведем лишь те атрибуты схемы, которые встретились во всех записях:

In [95]:
schema_fields_freqs[schema_fields_freqs['count'] == 1000]

,field,type,count
9,category,WSClassification,1000
16,externalIdSource,Primitive <string>,1000
15,externalId,Primitive <string>,1000
10,confidential,Primitive <boolean>,1000
22,info,WSContentInformation,1000
29,openAccessPermission,WSClassification,1000
32,personAssociations,Array <WSClassifiedAuthorAssociation>,1000
27,managingOrganisationalUnit,WSOrganisationRef,1000
25,language,WSClassification,1000
48,title,WSValue,1000


Таким образом, можно говорить о том, что каждая запись о результате научной деятельности будет содержать в себе:

- `category`, скорее всего тип деятельности
- `info`, метаданные
- `personAssociations`, привязка к людям
- `managingOrganisationalUnit`, ответственная организация
- `language`, скорее всего язык работы
- `title`, название
- `uuid`, уникальный идентификатор работы для взаимодействия с API
- `publicationStatuses`, скорее всего последовательность шагов для публикации
- `pureId`, внутренний идентификатор для Pure системы
- `totalNumberOfAuthors`, число авторов/участников

Эти атрибуты представляют меньший интерес:

- `type`, похоже на `category` согласно документации Pure, но четкая грань неясна
- `visibility`, скорее всего связано с тем, кто может просматривать контент
- `workflow`, Pure Workflows (описано в документации, пока неважно)

Остальное в целом можно проигнорировать в рамках ближайших исследований.

# Какие поля важны?

Не все поля нам нужны для какого-то первичного анализа.
К тому же все равно неясно, какие атрибуты будут присутствовать у каждой записи.

Поэтому, опираясь на свои представления о домене, отберем набор атрибутов, которые наверняка будут встречаться везде.

К таким можно отнести:

- `uuid` (*)
- `title` (*)
- `submissionYear` (нету)
- `type` (*)
- `category` (*)
- `managingOrganisationalUnit` (*)
- `personAssociations` (*)
- `electronicVersions`

Более сложные, не на сейчас:

- `keywordGroups`

### Получение интересующих значений

Вытащим обозначенные выше атрибуты в dataframe и посмотрим на результат

In [96]:
def get_electronic_version(obj):
    if 'electronicVersions' not in obj:
        return []
    
    result = []

    for version in obj['electronicVersions']:
        if 'doi' in version:
            result.append(version['doi'])
        if 'link' in version:
            result.append(version['link'])
    
    return result

def get_person_uuids(persons):
    result = []
    for person in persons:
        if 'person' in person:
            result.append(person['person']['uuid'])
        elif 'externalPerson' in person:
            result.append(f'Ext <{person['externalPerson']['uuid']}>')
    
    return result

def read_researches(research_list):
    result = []

    for research in research_list:
        research_filtered = {
            'uuid': research['uuid'],
            'title': research['title']['value'],
            'type': research['type']['pureId'],
            'category': research['category']['pureId'],
            'managing_org_unit': research['managingOrganisationalUnit']['uuid'],
            'contributors': get_person_uuids(research['personAssociations']),
            'electronicVersions': get_electronic_version(research)
        }

        result.append(research_filtered)
    
    return result

### Результат

In [97]:
filtered_researches = read_researches(researches)
researches_df = pd.DataFrame(filtered_researches)

Посмотрим, сколько исследований из выгрузки имеют хоть какие-то ссылки на сторонние источники

In [99]:
researches_df[researches_df['electronicVersions'].str.len() > 0].head()

,uuid,title,type,category,managing_org_unit,contributors,electronicVersions
5,9cd7de97-4eac-4f80-ae14-1d735c50113c,ЛЬВОВА Д.А. ПРОФЕССИОНАЛЬНЫЕ ОБЪЕДИНЕНИЯ БУХГА...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[5b9b301a-69c4-4bf1-b493-6fdc9f7ca918],[http://elibrary.ru/item.asp?id=11632741]
6,74bf478c-2d88-4aea-909d-3efcc6ae032b,"РЕЦЕНЗИЯ НА МОНОГРАФИЮ В.Ф. СЫЧА ""МОРФОЛОГИЯ Л...",4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[aed55111-b3f4-4cab-9542-aa5864aad36c],[http://elibrary.ru/item.asp?id=11845914]
8,34ab6333-82e9-4030-b824-cdf9c459d39d,ГОЛОВИН Н. А. ТЕОРЕТИКО-МЕТОДОЛОГИЧЕСКИЕ ОСНОВ...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[1c58e000-f429-4c46-bc79-efab3cc369b3],[http://elibrary.ru/item.asp?id=12867462]
9,9a4deab5-47a6-4f81-9209-12c88d44beaa,DAVIS R. TYPING POLITICS: THE ROLE OF BLOGS IN...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[cafe0010-4f6a-40c8-846f-cb2568c4686f],[http://elibrary.ru/item.asp?id=16973104]
12,37598330-9615-4e55-83e9-dc240c8adb72,МИФ ЦИФРОВОЙ ДЕМОКРАТИИ. РЕЦЕНЗИЯ НА КНИГУ: HI...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[cafe0010-4f6a-40c8-846f-cb2568c4686f],[http://elibrary.ru/item.asp?id=16401209]


`type` и `category` можно сопоставить по `pureId` с результатами, полученными в `classification.ipynb`.
Тогда можно будет более надежно определять, что это за работа; и важна ли она нам для наших конечных целей.